In [ ]:
WITH product_sales AS (
  SELECT 
    p.division AS division,
    p.product_code,
    p.product,
    SUM(s.sold_quantity) AS total_sold_quantity
  FROM 
    fact_sales_monthly s 
  JOIN 
    dim_product p ON s.product_code = p.product_code
  JOIN 
    fact_gross_price g ON s.product_code = g.product_code 
  AND
    s.fiscal_year = g.fiscal_year
  WHERE 
    s.fiscal_year = 2021
  GROUP BY 
    p.division, p.product_code, p.product),
ranked_products AS (
  SELECT 
    division,
    product_code,
    product,
    total_sold_quantity,
    DENSE_RANK() OVER (PARTITION BY division ORDER BY total_sold_quantity DESC) AS _rank
  FROM 
    product_sales)
  SELECT *
  FROM ranked_products
  WHERE 
    _rank IN (1, 2, 3);

In [ ]:
CREATE 
    ALGORITHM = UNDEFINED 
    DEFINER = `root`@`localhost` 
    SQL SECURITY DEFINER
VIEW `sales_post_inv_discount` AS
    SELECT 
        `s`.`date` AS `date`,
        `s`.`fiscal_year` AS `fiscal_year`,
        `s`.`customer_code` AS `customer_code`,
        `s`.`market` AS `market`,
        `s`.`product_code` AS `product_code`,
        `s`.`product` AS `product`,
        `s`.`variant` AS `variant`,
        `s`.`sold_quantity` AS `sold_quantity`,
        `s`.`gross_price_total` AS `gross_price_total`,
        `s`.`pre_invoice_discount_pct` AS `pre_invoice_discount_pct`,
        ((1 - `s`.`pre_invoice_discount_pct`) * `s`.`gross_price_total`) AS `net_invoice_sales`,
        (`po`.`discounts_pct` + `po`.`other_deductions_pct`) AS `post_invoice_discount_pct`
    FROM
        (`sales_pre_inv_discount` `s`
        JOIN `fact_post_invoice_deductions` `po` ON (((`s`.`date` = `po`.`date`)
            AND (`s`.`product_code` = `po`.`product_code`)
            AND (`s`.`customer_code` = `po`.`customer_code`))))

In [ ]:
CREATE DEFINER=`root`@`localhost` PROCEDURE `get_top_n_markets_in_each_region`(
in_fiscal_year int,
in_top_n int
)
BEGIN
with cte1 as (select c.market, c.region , round(sum(s.gross_price_total)/1000000,2) as gross_sales_mln
from gross_sales s
join dim_customer c
on s.customer_code = c.customer_code
where s.fiscal_year = in_fiscal_year
group by c.market, c.region)
,
cte2 as (select *, dense_rank() over(partition by region order by gross_sales_mln desc) as _rank
from cte1)

select * from cte2 where _rank <= in_top_n ;
END